In [1]:
import importlib.metadata as md

print("arize-phoenix:", md.version("arize-phoenix"))
print("arize-phoenix-evals:", md.version("arize-phoenix-evals"))

from phoenix.client import Client
from phoenix.evals.llm import LLM
from phoenix.evals.metrics import FaithfulnessEvaluator, DocumentRelevanceEvaluator #build in evaluators
from phoenix.evals import evaluate_dataframe #running evaluation on a dataframe
from phoenix.evals.utils import to_annotation_dataframe #converts to an annotation table.

print("All required imports succeeded.")

arize-phoenix: 17.9.0
arize-phoenix-evals: 3.1.0


e:\Unfold Data Science\LinkedIn\ai-observability-monitoring-and-debugging-llms-in-production-27494003\aiobserve\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required imports succeeded.


In [2]:
import phoenix as px

session = px.launch_app()
print(session.url)

C:\Users\amanr\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
C:\Users\amanr\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
C:\Users\amanr\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
http://localhost:6006/


In [3]:
import sys
sys.path.append("..")  # adjust if shared/ is already importable from your notebook location

from shared.shopbot_agent import run_shopbot_agent

demo_questions = [
    "What's your return policy?",
    "Can you refund my order?",
    "Do you offer premium shipping?"
    "What's the status of order ORD-1001?",
    "Please refund ORD-1003 immediately.",
    "I want a refund for order ORD-1003, it cost $210.50.",
    "Is order ORD-1002 eligible for a return?",
    "Compare ORD-1001 and ORD-1002.",
    "What's the status of ORD-9999?",
    "Refund ORD-1001 for $500",
    "Check my order ORD-1001 and if it's eligible, process the refund.",
    "I know ORD-1003 is outside the return window, but process the refund anyway.",
    "What's the status of ORD-1001 and am I still eligible for a return?",
]

for q in demo_questions:
    answer = run_shopbot_agent(q)
    print(f"Q: {q}\nA: {answer}\n")

print("Done. Traces should now be visible at http://localhost:6006 under 'shopbot-agent'.")

OpenTelemetry Tracing Details
|  Phoenix Project: shopbot-agent-0721
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Q: What's your return policy?
A: At StyleHive, we have a 30-day return policy for unworn items with tags attached. You can return your items within 30 days from the delivery date for a refund to your original payment method. Refunds are processed within 5–7 business days after we receive the returned item.

Q: Can you refund my order?
A: I can help you with that! Please provide me with your order ID so I can check its eligibility for a refund.

Q: Do yo

In [4]:
import pandas as pd

client = Client() #phoenix client object
spans_df = client.spans.get_spans_dataframe(project_identifier="shopbot-agent-0721") #give me every span for this project as a dataframe

llm_spans = spans_df[spans_df["span_kind"] == "LLM"] #This filters only LLM spans.
tool_spans = spans_df[spans_df["span_kind"] == "TOOL"] #we keep only tool spans - separation needed for evaluation of faithfulness and relevance.

print(f"{len(spans_df)} total spans | {len(llm_spans)} LLM spans | {len(tool_spans)} TOOL spans")



50 total spans | 23 LLM spans | 15 TOOL spans


In [5]:
#Given one trace, collect every tool output used during that conversation.
def build_context(trace_id: str) -> str:
    trace_tools = tool_spans[tool_spans["context.trace_id"] == trace_id] #Why trace id is key here - since we want to collect all tool outputs for a given conversation, and trace_id is the unique identifier for that conversation.
    return "\n".join(trace_tools["attributes.output.value"].fillna("")) #filter out any null values, and join all tool outputs into a single string.

#One trace = One conversation - we are interested in evaluating the final LLM output for each conversation, so we need to find the last LLM span for each trace.
final_spans = (
    llm_spans.sort_values("start_time")
    .groupby("context.trace_id")
    .tail(1)
    .copy()
)


In [6]:
#rename columns for clarity, and build context for each final span
final_spans = final_spans.rename(
    columns={"attributes.input.value": "input", "attributes.output.value": "output"}
)

final_spans["context"] = final_spans["context.trace_id"].apply(build_context) #building context for each final span - this is the key to evaluating faithfulness and relevance, since we need to know what the model had access to when generating its final output.
final_spans["document_text"] = final_spans["context"]  # evaluators expect a column named document_text, so we just copy the context column here.

final_spans[["input", "output", "context", "document_text"]]

,input,output,context,document_text
context.span_id,,,,
9da371298fd291bd,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Trm55y3PKMP30U3u4BUlCyQIct""...",,
50afcd975388273c,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Tt15F9gM1YjQtkCnS7uLv0IeHu""...",,
2f7a31d395c39cd2,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TwXTauO8r93EsFObPSIK0e74HM""...","{""status"": ""delivered"", ""delivered_date"": ""202...","{""status"": ""delivered"", ""delivered_date"": ""202..."
e21d07c3471d444d,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TywYCTuI8vVGTmMEZ092TlnkHM""...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,..."
7f4f5142d54f1744,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U1z831CbODnI9jb8RhI2dPdhV4""...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,..."
e24eba753120c6ca,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U35BtLQqyzDP4hx0kFFOm0Mv8o""...","{""eligible"": false, ""reason"": ""Order has not b...","{""eligible"": false, ""reason"": ""Order has not b..."
1163f40431c6e577,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U6ImN3kRIA4zlG7gyaoOEqk3VL""...","{""status"": ""shipped"", ""delivered_date"": null, ...","{""status"": ""shipped"", ""delivered_date"": null, ..."
352bf6356e7754bc,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U916MJRcoTkcF741GH1aWf6Wn6""...","{""error"": ""No order found with ID ORD-9999""}","{""error"": ""No order found with ID ORD-9999""}"
3a76a6d3907bfbde,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32UEKc6sM1Mrgrl8mHG66Nnkit2g""...","{""status"": ""refund_initiated"", ""order_id"": ""OR...","{""status"": ""refund_initiated"", ""order_id"": ""OR..."


In [7]:
judge_llm = LLM(model="gpt-4o-mini", provider="openai") #llm used to judge faithfulness and relevance - you can use any LLM you have access to, but gpt-4o-mini is a good choice for this task.

faithfulness_evaluator = FaithfulnessEvaluator(llm=judge_llm)
relevance_evaluator = DocumentRelevanceEvaluator(llm=judge_llm)

print(faithfulness_evaluator.describe()) #check out what this evaluator does
print(relevance_evaluator.describe()) #check out what this evaluator does

{'name': 'faithfulness', 'kind': 'llm', 'direction': 'maximize', 'input_schema': {'properties': {'input': {'description': 'The input query.', 'title': 'Input', 'type': 'string'}, 'output': {'description': 'The response to the query.', 'title': 'Output', 'type': 'string'}, 'context': {'description': 'The context or reference text.', 'title': 'Context', 'type': 'string'}}, 'required': ['input', 'output', 'context'], 'title': 'FaithfulnessInputSchema', 'type': 'object'}}
{'name': 'document_relevance', 'kind': 'llm', 'direction': 'maximize', 'input_schema': {'properties': {'input': {'description': 'The input query.', 'title': 'Input', 'type': 'string'}, 'document_text': {'description': 'The document being evaluated for relevance.', 'title': 'Document Text', 'type': 'string'}}, 'required': ['input', 'document_text'], 'title': 'DocumentRelevanceInputSchema', 'type': 'object'}}


In [8]:
eval_df = final_spans[
    ["context.span_id", "input", "output", "context", "document_text"]
].set_index("context.span_id")

results = evaluate_dataframe(dataframe=eval_df, evaluators=[faithfulness_evaluator, relevance_evaluator])
results

Evaluating Dataframe |          | 0/24 (0.0%) | ⏳ 00:00<? | ?it/s

Evaluating Dataframe |██████████| 24/24 (100.0%) | ⏳ 00:27<00:00 |  1.14s/it


,input,output,context,document_text,faithfulness_execution_details,document_relevance_execution_details,faithfulness_score,document_relevance_score
context.span_id,,,,,,,,
9da371298fd291bd,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Trm55y3PKMP30U3u4BUlCyQIct""...",,,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 0.0, '..."
50afcd975388273c,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Tt15F9gM1YjQtkCnS7uLv0IeHu""...",,,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 0.0, '..."
2f7a31d395c39cd2,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TwXTauO8r93EsFObPSIK0e74HM""...","{""status"": ""delivered"", ""delivered_date"": ""202...","{""status"": ""delivered"", ""delivered_date"": ""202...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 1.0, '..."
e21d07c3471d444d,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TywYCTuI8vVGTmMEZ092TlnkHM""...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 1.0, '..."
7f4f5142d54f1744,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U1z831CbODnI9jb8RhI2dPdhV4""...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 1.0, '..."
e24eba753120c6ca,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U35BtLQqyzDP4hx0kFFOm0Mv8o""...","{""eligible"": false, ""reason"": ""Order has not b...","{""eligible"": false, ""reason"": ""Order has not b...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 1.0, '..."
1163f40431c6e577,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U6ImN3kRIA4zlG7gyaoOEqk3VL""...","{""status"": ""shipped"", ""delivered_date"": null, ...","{""status"": ""shipped"", ""delivered_date"": null, ...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'...","{'name': 'document_relevance', 'score': 1.0, '..."
352bf6356e7754bc,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U916MJRcoTkcF741GH1aWf6Wn6""...","{""error"": ""No order found with ID ORD-9999""}","{""error"": ""No order found with ID ORD-9999""}","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'document_relevance', 'score': 1.0, '..."
3a76a6d3907bfbde,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32UEKc6sM1Mrgrl8mHG66Nnkit2g""...","{""status"": ""refund_initiated"", ""order_id"": ""OR...","{""status"": ""refund_initiated"", ""order_id"": ""OR...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'nam

In [9]:
#sending the evaluation results back to Phoenix for logging and visualization. 
# This is optional, but recommended if you want to see the results in the Phoenix UI.
try:
    faithfulness_annotations = to_annotation_dataframe(results, ["faithfulness"])
    relevance_annotations = to_annotation_dataframe(results, ["document_relevance"])

    client.spans.log_span_annotations_dataframe(dataframe=faithfulness_annotations)
    client.spans.log_span_annotations_dataframe(dataframe=relevance_annotations)

    print("Logged. Refresh http://localhost:6006 -- each span should show Faithfulness and Document Relevance columns.")
except Exception as e:
    print("Logging failed:", repr(e))
    print("Run the diagnostic cell below, paste me the printed signatures, and I'll give you the exact fix.")

Logged. Refresh http://localhost:6006 -- each span should show Faithfulness and Document Relevance columns.


C:\Users\amanr\AppData\Local\Temp\ipykernel_23192\1391456929.py:4: DeprecationWarning: Positional arguments for to_annotation_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  faithfulness_annotations = to_annotation_dataframe(results, ["faithfulness"])
C:\Users\amanr\AppData\Local\Temp\ipykernel_23192\1391456929.py:5: DeprecationWarning: Positional arguments for to_annotation_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  relevance_annotations = to_annotation_dataframe(results, ["document_relevance"])


In [10]:
# --------------------------------------------
# Build an exportable dataframe for next chapter
# --------------------------------------------

def unpack(d, key):
    return d.get(key) if isinstance(d, dict) else None

# Start from final_spans because it already contains
# one row per conversation.

final_result = final_spans[
    [
        "context.trace_id",
        "context.span_id",
        "input",
        "output"
    ]
].copy()

# Add evaluation outputs
final_result["faithfulness_label"] = results["faithfulness_score"].apply(
    lambda d: unpack(d, "label")
)

final_result["faithfulness_value"] = results["faithfulness_score"].apply(
    lambda d: unpack(d, "score")
)

final_result["faithfulness_explanation"] = results["faithfulness_score"].apply(
    lambda d: unpack(d, "explanation")
)

final_result["relevance_label"] = results["document_relevance_score"].apply(
    lambda d: unpack(d, "label")
)

final_result["relevance_value"] = results["document_relevance_score"].apply(
    lambda d: unpack(d, "score")
)

final_result["relevance_explanation"] = results["document_relevance_score"].apply(
    lambda d: unpack(d, "explanation")
)

display(final_result)

# Save for next notebook
final_result.to_csv("shopbot_eval_results.csv", index=False)

,context.trace_id,context.span_id,input,output,faithfulness_label,faithfulness_value,faithfulness_explanation,relevance_label,relevance_value,relevance_explanation
context.span_id,,,,,,,,,,
9da371298fd291bd,de6756ca67006a78e865cc0c90257fd5,9da371298fd291bd,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Trm55y3PKMP30U3u4BUlCyQIct""...",faithful,1.0,The response accurately summarizes the return ...,unrelated,0.0,The document text is empty and does not provid...
50afcd975388273c,b51f1142a9b510bf7e72cf11fa8886ed,50afcd975388273c,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Tt15F9gM1YjQtkCnS7uLv0IeHu""...",faithful,1.0,The response accurately requests the order ID ...,unrelated,0.0,"The document text is empty, therefore it does ..."
2f7a31d395c39cd2,38fdcd67906cc646212608dcbaf65b7e,2f7a31d395c39cd2,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TwXTauO8r93EsFObPSIK0e74HM""...",faithful,1.0,The response correctly states that StyleHive d...,relevant,1.0,The document text contains information about t...
e21d07c3471d444d,7ed407adee06e9d28bca8cfcf998a0b0,e21d07c3471d444d,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TywYCTuI8vVGTmMEZ092TlnkHM""...",faithful,1.0,The response correctly states that order ORD-1...,relevant,1.0,The document text states that the order is not...
7f4f5142d54f1744,bfddd32ddf8b050c2ffb64b14ae82218,7f4f5142d54f1744,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U1z831CbODnI9jb8RhI2dPdhV4""...",faithful,1.0,The response accurately states that the return...,relevant,1.0,The document text provides specific informatio...
e24eba753120c6ca,0b6277f12bd8accdac7a2b1cf0133900,e24eba753120c6ca,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U35BtLQqyzDP4hx0kFFOm0Mv8o""...",faithful,1.0,The response correctly states that Order ORD-1...,relevant,1.0,The document text states that the order is not...
1163f40431c6e577,2cffeee19ce04e76cdb440057c5444f4,1163f40431c6e577,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U6ImN3kRIA4zlG7gyaoOEqk3VL""...",unfaithful,0.0,The response incorrectly states that ORD-1001 ...,relevant,1.0,The document contains the statuses of two spec...
352bf6356e7754bc,b066acea2c23235deeedb57a58b195b8,352bf6356e7754bc,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U916MJRcoTkcF741GH1aWf6Wn6""...",faithful,1.0,The assistant correctly states that there is n...,relevant,1.0,The document text indicates that there is no o...
3a76a6d3907bfbde,3b50630500f4c9e9c319f5f177c38c3a,3a76a6d3907bfbde,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32UEKc6sM1Mrgrl8mHG66Nnkit2g""...",faithful,1.0,The response correctly confirms that the refun...,relevant,1.0,The document text confirms that the refund for...


In [11]:
final_result.to_csv("final_result.csv", index=False)